# SARSA（State-Action-Reward-State-Action）算法（在线策略时序差分控制）

> SARSA 也是 **Robbins-Monro（RM）随机近似**算法的应用，与 TD(0) 的区别在于：求解目标从**状态价値 $V(s)$** 换成了**动作价値 $Q(s,a)$**。
>
> **TD(0)** 求解贝尔曼期望方程（状态价値）：
>
> $$\mathbb{E}\bigl[V(s_t) - (r_{t+1} + \gamma V(s_{t+1}))\bigr] = 0$$
>
> $$V(s_t) \leftarrow V(s_t) - \alpha \cdot \underbrace{\bigl[V(s_t) - (r_{t+1} + \gamma V(s_{t+1}))\bigr]}_{\text{TD 误差，用 }(s,r,s')}$$
>
> **SARSA** 求解贝尔曼期望方程（**动作**价値）：
>
> $$\mathbb{E}\bigl[Q(s_t,a_t) - (r_{t+1} + \gamma Q(s_{t+1},a_{t+1}))\bigr] = 0$$
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \cdot \underbrace{\bigl[Q(s_t,a_t) - (r_{t+1} + \gamma Q(s_{t+1},a_{t+1}))\bigr]}_{\text{TD 误差，用 }(s,a,r,s',a')}$$
>
> 两者在 RM 框架下的对比如下：
>
> ---

| RM 符号 | TD(0) 中的含义 | SARSA 中的含义 |
|---|---|---|
| $\theta_k$ | $V(s_t)$ | $Q(s_t, a_t)$ |
| $g(\theta_k, \eta_k)$ | $V(s_t) - (r_{t+1} + \gamma V(s_{t+1}))$ | $Q(s_t,a_t) - (r_{t+1} + \gamma Q(s_{t+1},a_{t+1}))$ |
| 所需样本 | $(s, r, s')$ 三元组 | $(s, a, r, s', a')$ 五元组 |

> ---
>
> 这正是 SARSA 名字的由来——**State-Action-Reward-State-Action**，每次更新需要五元组，比 TD(0) 多出了 $a_t$ 和 $a_{t+1}$ 两个动作信息。

## 一、导入库与环境初始化

In [1]:
# 导入 NumPy 库，用于数组操作（如 Q 表初始化、策略矩阵构建等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块，可用于控制训练展示速率（睡眠等待）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 由于环境文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                         "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location("GridWorld_v2", _env_path)  # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [2]:
# 设置网格世界的行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界的列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚奖励
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图
gridworld.show()

# 初始化状态价值向量（全零）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计，初始全为 0
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作[上下左右原地]的 Q 值
# 初始全零很重要，否则会引导智能体优先访问没去过的状态（乐观初始化）
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、SARSA 算法实现

> **On-Policy（在线策略）** 是指：用来**采样**的策略与被**评估改进**的策略是同一个。
>
> SARSA 的 Q 値更新公式：
>
> $$Q(s_t, a_t) \leftarrow Q(s_t, a_t) - \alpha \left[Q(s_t, a_t) - (r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}))\right]$$
>
> 其中 $a_{t+1}$ 是智能体按当前策略（epsilon-greedy）**实际执行**的下一动作，而非理论最优动作。
>
> **为什么这是 On-Policy？**
>
> - 采样策略（轨迹生成）：epsilon-greedy 策略 $\pi$
> - 被评估的策略（Q 更新目标）：$r_{t+1} + \gamma Q(s_{t+1}, a_{t+1})$，其中 $a_{t+1} \sim \pi$，也是同一个 $\pi$
>
> 两者始终是同一个策略，所以是 On-Policy。相比之下，Q-Learning 在计算目标时用 $\max_a Q(s_{t+1}, a)$（最优动作）而非实际执行的动作，因此是 Off-Policy。

In [3]:
def SARSA(gridworld: GridWorld_v2.GridWorld_v2,
         gamma=0.99,
         trajectorySteps=-1,
         learning_rate=0.001,
         final_epsilon=0.01,
         num_episodes=600) -> GridWorld_v2.GridWorld_v2:
    '''
    SARSA 算法（State-Action-Reward-State-Action）。

    On-Policy 的 TD 控制算法，用行为策略（epsilon-greedy）
    同时进行策略评估和策略改进。

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        gamma (float): 折扣因子，衡量未来奖励的重要程度，取值 (0,1]
        trajectorySteps (int): 每条轨迹的最大步数；
                               -1 表示到达目标后自动停止
        learning_rate (float): TD 学习率（步长），控制 Q 值更新幅度
        final_epsilon (float): epsilon-greedy 中 epsilon 的最小值，
                               防止完全停止探索（0~1）
        num_episodes (int): 训练轮数（每轮生成一条轨迹）

    返回值:
        np.ndarray: 最终收敛的动作价值矩阵（Q 表），
                    shape (25, 5)——行=状态，列=动作
    '''
    # 初始化状态价值向量（全零，仅用于计算展示）
    # shape: (25,)——每个元素对应一个状态的价值
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
    action_value = np.zeros((rows * columns, 5))

    # 初始化随机确定性策略（独热编码）
    # np.random.randint(0,5, size=(25,))：每个状态随机选一个动作，shape (25,)
    # np.eye(5)[...]：转为独热向量，shape (25, 5)——每行只有一个位置为 1
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    epsilon = 0.5  # 初始 epsilon 值（较高，鼓励早期大量探索）

    for episode in range(num_episodes):  # 训练主循环，共 num_episodes 轮
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次进度

        # 线性衰减 epsilon（逐步从探索转向利用）
        if epsilon > final_epsilon:
            epsilon -= 0.001   # 每轮减少 0.001
        else:
            epsilon = final_epsilon  # 不低于最小探索率

        # 计算 epsilon-greedy 策略的概率分配
        # p1：贪心动作（策略中值为1的动作）的选取概率
        p1 = 1 - epsilon * (4/5)
        # p0：非贪心动作（策略中值为0的动作）的选取概率（共4种均分 epsilon）
        p0 = epsilon / 5
        # 概率映射字典：策略矩阵中 1→p1，0→p0
        d = {1: p1, 0: p0}
        print("p1", p1, "p0", p0)  # 打印当前轮次的概率值

        # 将确定性策略（独热）转为 epsilon-greedy 概率分布
        # 返回值 shape: (25, 5)——每行是一个状态的动作概率分布
        policy_epsilon = np.vectorize(d.get)(policy)

        # 初始化状态访问计数器（长度 25 的列表）
        cnt = [0 for i in range(25)]

        # 固定初始状态（状态10，位于地图中部）
        initState = 10
        # 随机选择初始动作（0~4 的整数）
        initAction = random.randint(0, 4)

        if trajectorySteps == -1:
            stop_when_reach_target = True  # 到达目标自动停止

        # 按 epsilon-greedy 策略生成一条轨迹
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,
            action=initAction,
            policy=policy_epsilon,
            steps=trajectorySteps,
            stop_when_reach_target=True)

        # 在轨迹末尾追加目标格自循环转移，确保终止状态价值稳定
        # (17, 4, 1, 17, 4)：状态17（目标）执行动作4，奖励1，保持在状态17
        Trajectory.append((17, 4, 1, 17, 4))
        print("trajectorySteps", len(Trajectory))  # 打印轨迹长度

        steps = len(Trajectory) - 1  # 实际有效步数

        # 从轨迹末尾向前遍历，执行 SARSA 的 TD 更新
        for k in range(steps, -1, -1):
            # 解包当前步转移元组
            # tmpstate:  当前状态索引 int
            # tmpaction: 当前动作索引 int
            # tmpscore:  即时奖励 float
            # nextState: 下一状态索引 int
            # nextAction:下一动作索引 int（SARSA 关键：用实际执行的下一动作！）
            tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
            cnt[tmpstate] += 1  # 累加状态访问次数

            # SARSA TD 误差：Q(s,a) - [r + γ*Q(s',a')]
            # 与 Q-Learning 的区别：使用实际的下一动作 a' 而非最优动作
            TD_error = (action_value[tmpstate][tmpaction]
                        - (tmpscore + gamma * action_value[nextState][nextAction]))
            # 按 TD 误差更新 Q 值（朝减小误差方向修正）
            action_value[tmpstate][tmpaction] -= learning_rate * TD_error

        # 策略改进：对每个状态选择 Q 值最大的动作作为新的贪心策略
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态的最优动作索引
        # np.eye(5)[...]：转为独热编码，shape (25, 5)
        policy = np.eye(5)[np.argmax(action_value, axis=1)]
        # 更新 epsilon-greedy 策略供下轮采样使用
        policy_epsilon = np.vectorize(d.get)(policy)

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算当前策略下的状态价值（用于可视化进度）
        # V(s) = Σ_a π(a|s) * Q(s,a)，对每个状态按策略概率加权 Q 值
        # shape: (25,)——每个状态的估计价值
        state_value = np.sum(policy_epsilon * action_value, axis=1)
        # 计算全局平均状态价值（衡量整体策略质量）
        mean_state_value = state_value.mean()

        gridworld.showPolicy(policy)                                   # 显示当前策略
        print(np.round(state_value, decimals=4).reshape(5, 5))        # 打印 5×5 状态价值矩阵（保留4位小数）
        print("mean_state_value", mean_state_value)                    # 打印平均状态价值

    return action_value  # 返回最终学到的动作价值矩阵（Q 表），shape (25, 5)

## 三、运行 SARSA 算法并查看结果

In [4]:
# 调用 SARSA 算法，使用默认超参数训练智能体
# 返回值：action_value，shape (25, 5)——最终收敛的 Q 表
action_value = SARSA(gridworld)

episode 0/600
p1 0.6008 p0 0.0998
trajectorySteps 177
[[50 36  5  0  0]
 [10 26  5  5  0]
 [ 6  8  0  0  0]
 [ 1  6  2  0  0]
 [11  4  2  0  0]]
➡️➡️➡️⬆️⬆️
⬇️⏪⏬⬆️⬆️
➡️➡️⏫️⬆️⬆️
➡️⏩️✅⏫️⬆️
➡️⏫️⬆️⬆️⬆️
[[-0.0031 -0.0033 -0.0001  0.      0.    ]
 [-0.0032 -0.0199 -0.     -0.004   0.    ]
 [-0.0001 -0.006   0.      0.      0.    ]
 [-0.     -0.003  -0.0004  0.      0.    ]
 [-0.001  -0.002   0.0005  0.      0.    ]]
mean_state_value -0.0018263388516062248
episode 1/600
p1 0.6015999999999999 p0 0.0996
trajectorySteps 6
[[0 0 0 0 0]
 [0 0 0 0 0]
 [1 0 0 0 0]
 [1 2 2 0 0]
 [0 0 0 0 0]]
➡️➡️➡️⬆️⬆️
⬇️⏪⏬⬆️⬆️
➡️➡️⏫️⬆️⬆️
⬇️⏩️✅⏫️⬆️
➡️⏫️⬆️⬆️⬆️
[[-0.0031 -0.0033 -0.0001  0.      0.    ]
 [-0.0032 -0.0199 -0.     -0.004   0.    ]
 [-0.0001 -0.006   0.      0.      0.    ]
 [-0.001  -0.0034 -0.0008  0.      0.    ]
 [-0.001  -0.002   0.0005  0.      0.    ]]
mean_state_value -0.0018942649360170216
episode 2/600
p1 0.6024 p0 0.0994
trajectorySteps 421
[[  8   8  24 132 116]
 [  9   5  32  22  14]
 [  7   8

In [5]:
# 显示最终学到的动作价值矩阵（Q 表）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
action_value

array([[-2.06288441e-01, -1.57638235e-02, -1.70931974e-02,
        -1.80912267e-01, -2.43258990e-02],
       [-9.80035987e-02, -6.66703731e-03, -8.08426005e-01,
        -8.61768033e-03, -8.72044957e-03],
       [-5.46257741e-02, -2.35070927e-03, -5.17828431e-01,
        -2.89422361e-03, -2.58256765e-03],
       [-1.62613728e-01, -1.88644982e-03, -2.08446123e-03,
        -3.17271304e-03, -5.88740956e-03],
       [-1.12483166e-01, -4.65197224e-02, -1.38319429e-03,
        -1.50059771e-03, -1.39275685e-03],
       [-9.53522340e-03, -1.05612369e+00, -1.27956807e-02,
        -9.02343129e-02, -1.20706467e-02],
       [-2.63920331e-03, -2.08396178e-01, -2.68297896e-03,
        -2.64572966e-03, -4.23078072e-01],
       [-5.09254223e-04, -2.83605738e-04, -2.87599156e-01,
        -1.19435146e-01, -8.03709527e-02],
       [-1.63811973e-03, -1.05564781e-03, -1.09187336e-03,
        -3.54880195e-01, -1.50486209e-03],
       [-1.57623127e-03, -1.10190429e-01, -1.47612374e-03,
        -1.60418935e-03